# Data observability: Crown Seaways and Pearl Seaways AIS data

Measures how well the data meets our expectations (required columns, valid values, etc.).
The same checks run on the **raw** data and on the **cleaned** data, so we can show what cleaning fixed.

## Step 1: Load the raw data

In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Load the file **as is**, with all columns and no cleaning. The timestamp is read as text and then parsed with `errors='coerce'`, so unparseable values become `NaT` and get counted instead of stopping the load.

Columns with no information (`geom`, `geometry_wkt`, etc.) are kept here so the checks can report them; they are dropped in the cleaning notebook.

In [8]:
DATA_PATH = Path('..', 'data', 'ais_221026-ramleh.csv')

raw = pd.read_csv(DATA_PATH, low_memory=False)
raw['date_time_utc'] = pd.to_datetime(raw['date_time_utc'], errors='coerce')

print(f'{len(raw):,} rows, {raw.shape[1]} columns, '
      f'{raw.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory')
raw.head()

3,134,835 rows, 22 columns, 1743 MB in memory


,date_time_utc,mmsi,dsrc,imo,ship_name,ship_type,callsign,maneuvre,lon,lat,...,rot,speed,cog,true_heading,length,breadth,draught,ais_class,geometry_wkt,date_utc
0,2025-01-01 11:06:43,219592000,S,8917613,CROWN SEAWAYS,60,OXRA6,NaN,12.596666,55.701666,...,NaN,0.0,203.0,NaN,171,28,6.2,LRIT,POINT(12.596666 55.701666),2025-01-01
1,2025-01-01 00:03:49,219592000,G,8917613,CROWN SEAWAYS,60,OXRA6,0.0,10.831520,58.384247,...,11.0,16.9,159.4,159.0,171,28,6.2,A,POINT(10.83152 58.38424666),2025-01-01
2,2025-01-01 03:02:23,219945000,G,8701674,PEARL SEAWAYS,69,OWFU2,0.0,10.943467,58.145100,...,-8.0,16.2,355.0,353.0,178,30,6.4,A,POINT(10.94346666 58.1451),2025-01-01
3,2025-01-01 03:51:35,219945000,G,8701674,PEARL SEAWAYS,69,OWFU2,0.0,10.879617,58.365367,...,0.0,16.0,349.5,351.0,178,30,6.4,A,POINT(10.87961666 58.36536666),2025-01-01
4,2025-01-01 04:35:59,219945000,G,8701674,PEARL SEAWAYS,69,OWFU2,0.0,10.820750,58.568717,...,13.0,16.6,355.0,356.0,178,30,6.4,A,POINT(10.82075 58.56871666),2025-01-01


In [9]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3134835 entries, 0 to 3134834
Data columns (total 22 columns):
 #   Column         Dtype         
---  ------         -----         
 0   date_time_utc  datetime64[us]
 1   mmsi           int64         
 2   dsrc           str           
 3   imo            int64         
 4   ship_name      str           
 5   ship_type      int64         
 6   callsign       str           
 7   maneuvre       float64       
 8   lon            float64       
 9   lat            float64       
 10  geom           str           
 11  status         int64         
 12  rot            float64       
 13  speed          float64       
 14  cog            float64       
 15  true_heading   float64       
 16  length         int64         
 17  breadth        int64         
 18  draught        float64       
 19  ais_class      str           
 20  geometry_wkt   str           
 21  date_utc       str           
dtypes: datetime64[us](1), float64(8), int64(6), str(7)
me

The raw data has 22 columns. Three of them are dropped in the cleaning notebook, because they add no information that is not already in another column:

- **maneuvre**: always 0 or missing, so it does not tell us anything.
- **geom / geometry_wkt**: the position written as text, redundant since we get the position from lon/lat.

Which of the remaining 19 columns we need is decided in the next step, where we list our required columns.

**Types**: pandas read each column with a type that fits its content. `date_time_utc` is a datetime (`datetime64[us]`), `mmsi` and `imo` are integers, the positions, speed, course, heading and draught are floats, and the text columns (`ship_name`, `ais_class`) are strings (`str`).